# LoRA-Tuned UNI Feature Extraction for PANDA (Stage 3) -- FINAL (v5 fix-pass)

**All known bugs fixed (cumulative through v5):**
- v3: HuggingFace PEFT loading via plain `load_state_dict(strict=False)`
- v3: strong sanity check (sample LoRA tensor changed AND matches checkpoint)
- v3: pre-flight slide verification removed (CPU workers have local-copy fallback)
- v3: CPU workers retry slide open up to 3 times with copy-to-local-SSD on retry
- **v5: patch_size 256 -> 224 (matches LoRA training distribution; no GPU-side resize)**
- **v5: HF_TOKEN read from Colab secret first, hardcoded literal as fallback only**

**Workflow:** Held-out fold per slide. Slides in fold N encoded by `lora_fold{N}_best.pt`. Compatible with main_v6/v8 Stage 4.

**Why use this instead of v8's built-in `run_stage_3()`:** v8's Stage 3 is single-threaded and encodes every slide with a single best-fold checkpoint (which leaks data because some slides were in that fold's training set). This notebook (v4/v5) is multi-threaded with 5 CPU workers feeding the GPU (5-10x faster on A100) AND uses the correct held-out fold per slide.

**Before running:** set the `HF_TOKEN` secret in Colab (key-icon panel, left sidebar). Install dependencies will run in cell 2; restart the runtime once after the install completes.


## 1. Setup & Imports

In [ ]:
# ── CLAM-TR LoRA v1: Environment Setup ──
import subprocess

# Install system dependencies
for pkg in ['openslide-tools', 'libgl1-mesa-glx']:
    subprocess.run(['apt-get', 'install', '-y', '-q', pkg], capture_output=True)

# Uninstall Colab's incompatible torchao (PEFT requires >= 0.16, Colab ships 0.10)
subprocess.run(['pip', 'uninstall', '-y', '-q', 'torchao'], capture_output=True)

# Install Python packages (peft for LoRA)
for pkg in ['openslide-python', 'timm', 'huggingface_hub', 'h5py',
            'opencv-python-headless', 'peft', 'psutil']:
    subprocess.run(['pip', 'install', '-q', pkg], capture_output=True)

# Mount Google Drive (same way the training notebook does it)
from google.colab import drive
drive.mount('/content/drive')

# ── Imports ──
import os
import json
import time
import threading
import queue
import shutil
import numpy as np
from datetime import datetime
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms
from tqdm import tqdm
import timm

import cv2
import openslide

from huggingface_hub import hf_hub_download, login as hf_login
from peft import LoraConfig, get_peft_model

# ── GPU ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── HuggingFace Auth (UNI is gated) ──
# HF_TOKEN: prefer Colab secret (set in Colab key-icon panel as HF_TOKEN);
# fall back to a hardcoded literal only if the secret is unavailable.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = "hf_YOUR_TOKEN_HERE"  # <-- replace if you are not using Colab secrets
hf_login(token=HF_TOKEN)
print("HuggingFace: Logged in")

print("Setup complete.")


## 2. Configuration

In [2]:
# ══════════════════════════════════════════════════
# EDIT THESE PATHS FOR YOUR SETUP
# ══════════════════════════════════════════════════

SLIDES_DIR = '/content/drive/MyDrive/PANDA_Subset_9128/images/'
OUTPUT_DIR = '/content/drive/MyDrive/PANDA_LoRA/lora_features/'
PROGRESS_DIR = '/content/drive/MyDrive/PANDA_LoRA/progress/'
LORA_CHECKPOINTS_DIR = '/content/drive/MyDrive/PANDA_LoRA/checkpoints/'
LORA_CKPT_PATTERN = 'lora_fold{fold}_best.pt'
LABELS_PATH = '/content/drive/MyDrive/panda-data/prostate-cancer-grade-assessment/Kaggle-PANDA-1st-place-solution-master/input/train-5kfold_remove_noisy_by_0622_rad_13_08_ka_15_10.csv'

# ══════════════════════════════════════════════════
# LoRA CONFIG — MUST MATCH TRAINING (read from checkpoint to verify)
# ══════════════════════════════════════════════════
# From your checkpoint's lora_config:
#   r=8, lora_alpha=16, lora_dropout=0.05, target_modules=['qkv'], bias='none'
LORA_CONFIG = {
    'r': 8,
    'lora_alpha': 16,
    'lora_dropout': 0.05,
    'target_modules': ['qkv'],
    'bias': 'none',
}

# ══════════════════════════════════════════════════
# PIPELINE SETTINGS
# ══════════════════════════════════════════════════

EXTRACTION_CONFIG = {
    'slides_dir': SLIDES_DIR,
    'slide_ext': '.tiff',
    'output_dir': OUTPUT_DIR,
    'progress_dir': PROGRESS_DIR,
    'patch_size': 224,  # v5: matches LoRA training distribution; drop the GPU-side resize
    'target_magnification': 20,
    'encoder_batch_size': 256,
    'checkpoint_every': 50,
    'min_tissue_ratio': 0.5,
    'sat_thresh': 8,
    'median_blur': 7,
    'morph_close': 4,
    'white_sat_thresh': 5,
    'black_rgb_thresh': 40,
    'min_patches_per_slide': 10,
    'n_workers': 5,
    'retry_failed': True,
    'local_temp_dir': '/content/local_slides_temp',
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PROGRESS_DIR, exist_ok=True)

import pandas as pd
missing_ckpts = []
for fold in range(5):
    p = os.path.join(LORA_CHECKPOINTS_DIR, LORA_CKPT_PATTERN.format(fold=fold))
    if not os.path.exists(p):
        missing_ckpts.append(fold)
if missing_ckpts:
    print(f"WARNING: Missing LoRA checkpoints for folds: {missing_ckpts}")
    print(f"Expected at: {LORA_CHECKPOINTS_DIR}")
else:
    print(f"All 5 LoRA checkpoints found at {LORA_CHECKPOINTS_DIR}")

labels_df = pd.read_csv(LABELS_PATH)
if labels_df['kfold'].min() == 1:
    labels_df['kfold'] = labels_df['kfold'] - 1
print(f"Labels loaded: {len(labels_df)} slides, folds {sorted(labels_df['kfold'].unique())}")
print(f"Per-fold counts: {labels_df.groupby('kfold').size().to_dict()}")

if os.path.isdir(SLIDES_DIR):
    slide_files = [f for f in os.listdir(SLIDES_DIR) if f.endswith('.tiff')]
    print(f"\nFound {len(slide_files)} .tiff slides")
    existing = len([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.pt')]) if os.path.isdir(OUTPUT_DIR) else 0
    print(f"Existing LoRA features: {existing}")
    print(f"Remaining: ~{len(slide_files) - existing}")
else:
    print(f"\nWARNING: Slides not found at {SLIDES_DIR}")


Configuration:
  slides_dir: /content/drive/MyDrive/PANDA_Subset_9128/images/
  slide_ext: .tiff
  output_dir: /content/drive/MyDrive/UNI_features/
  progress_dir: /content/drive/MyDrive/CLAM_TR_Preprocessing/v20/progress
  patch_size: 256
  target_magnification: 20
  encoder_batch_size: 256
  checkpoint_every: 50
  min_tissue_ratio: 0.5
  sat_thresh: 8
  median_blur: 7
  morph_close: 4
  white_sat_thresh: 5
  black_rgb_thresh: 40
  min_patches_per_slide: 10
  n_workers: 5
  retry_failed: True
  local_temp_dir: /content/local_slides_temp

Found 9128 slides in /content/drive/MyDrive/PANDA_Subset_9128/images/
Existing features: 9128
Remaining: ~0


## 3. Extraction Pipeline

Same multi-worker pipeline as the UNI extraction notebook, but the encoder is now a LoRA-wrapped UNI. We process slides **fold-by-fold**: load fold 0's LoRA, encode all fold 0 slides, then swap to fold 1's LoRA, etc. Only 5 LoRA loads total.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Slide Preprocessor — IDENTICAL to UNI extraction
# ══════════════════════════════════════════════════════════════════

class SlidePreprocessor:
    """HSV tissue seg + magnification normalization + white/black filtering."""
    def __init__(self, patch_size=256, target_mag=20, min_tissue_ratio=0.5,
                 sat_thresh=8, median_blur=7, morph_close=4,
                 white_sat_thresh=5, black_rgb_thresh=40):
        self.patch_size = patch_size
        self.target_mag = target_mag
        self.min_tissue_ratio = min_tissue_ratio
        self.sat_thresh = sat_thresh
        self.median_blur = median_blur
        self.morph_close = morph_close
        self.white_sat_thresh = white_sat_thresh
        self.black_rgb_thresh = black_rgb_thresh

    def get_extraction_params(self, slide):
        try:
            obj_power = float(slide.properties.get('openslide.objective-power', 20))
        except (ValueError, KeyError):
            obj_power = 20
        downsample_needed = obj_power / self.target_mag
        best_level, best_diff = 0, float('inf')
        for level in range(slide.level_count):
            diff = abs(slide.level_downsamples[level] - downsample_needed)
            if diff < best_diff:
                best_diff, best_level = diff, level
        level_ds = slide.level_downsamples[best_level]
        actual_mag = obj_power / level_ds
        mag_ratio = actual_mag / self.target_mag
        read_size = int(round(self.patch_size * mag_ratio))
        step_l0 = int(round(read_size * level_ds))
        return best_level, read_size, step_l0, actual_mag, obj_power

    def segment_tissue(self, slide):
        seg_level = 0
        for level in range(slide.level_count):
            if slide.level_downsamples[level] <= 64:
                seg_level = level
        dims = slide.level_dimensions[seg_level]
        img = np.array(slide.read_region((0, 0), seg_level, dims).convert('RGB'))
        img_hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        img_med = cv2.medianBlur(img_hsv[:, :, 1], self.median_blur)
        _, tissue_mask = cv2.threshold(img_med, self.sat_thresh, 255, cv2.THRESH_BINARY)
        if self.morph_close > 0:
            kernel = np.ones((self.morph_close, self.morph_close), np.uint8)
            tissue_mask = cv2.morphologyEx(tissue_mask, cv2.MORPH_CLOSE, kernel)
        return tissue_mask, seg_level

    def extract_patches_and_coords(self, slide):
        level, read_size, step_l0, _, _ = self.get_extraction_params(slide)
        tissue_mask, seg_level = self.segment_tissue(slide)
        seg_ds = slide.level_downsamples[seg_level]
        mask_h, mask_w = tissue_mask.shape
        needs_resize = (read_size != self.patch_size)
        patches, coords = [], []
        w0, h0 = slide.level_dimensions[0]
        for y in range(0, h0 - step_l0 + 1, step_l0):
            for x in range(0, w0 - step_l0 + 1, step_l0):
                mx = min(int(x / seg_ds), mask_w - 1)
                my = min(int(y / seg_ds), mask_h - 1)
                mw = max(1, min(int(step_l0 / seg_ds), mask_w - mx))
                mh = max(1, min(int(step_l0 / seg_ds), mask_h - my))
                if mw <= 0 or mh <= 0:
                    continue
                tissue_ratio = np.mean(tissue_mask[my:my+mh, mx:mx+mw] > 0)
                if tissue_ratio < self.min_tissue_ratio:
                    continue
                try:
                    patch_pil = slide.read_region((x, y), level, (read_size, read_size)).convert('RGB')
                    patch_arr = np.array(patch_pil)
                    patch_hsv = cv2.cvtColor(patch_arr, cv2.COLOR_RGB2HSV)
                    if np.mean(patch_hsv[:, :, 1]) < self.white_sat_thresh:
                        continue
                    if np.all(np.mean(patch_arr, axis=(0, 1)) < self.black_rgb_thresh):
                        continue
                    if needs_resize:
                        patch_pil = patch_pil.resize((self.patch_size, self.patch_size), Image.LANCZOS)
                        patch_arr = np.array(patch_pil)
                    patches.append(patch_arr)
                    coords.append([x, y])
                except Exception:
                    continue
        return patches, np.array(coords, dtype=np.int32) if coords else np.zeros((0, 2), dtype=np.int32)


# ══════════════════════════════════════════════════════════════════
# UNI v1 + PEFT LoRA Encoder (matches training exactly)
# ══════════════════════════════════════════════════════════════════

def build_uni_with_peft_lora(lora_cfg):
    """Load UNI v1 base + wrap with PEFT LoRA. Same as the training notebook."""
    print("Loading UNI v1 (ViT-L/16)...")
    base = timm.create_model(
        "vit_large_patch16_224",
        img_size=224, patch_size=16,
        init_values=1e-5, num_classes=0, dynamic_img_size=True,
    )
    weights_path = hf_hub_download("MahmoodLab/UNI", "pytorch_model.bin")
    base.load_state_dict(
        torch.load(weights_path, map_location="cpu", weights_only=False), strict=True)
    for p in base.parameters():
        p.requires_grad = False

    peft_cfg = LoraConfig(
        r=lora_cfg['r'],
        lora_alpha=lora_cfg['lora_alpha'],
        lora_dropout=lora_cfg['lora_dropout'],
        target_modules=lora_cfg['target_modules'],
        bias=lora_cfg['bias'],
    )
    model = get_peft_model(base, peft_cfg)
    # Count params
    n_total = sum(p.numel() for p in model.parameters())
    n_lora = sum(p.numel() for n, p in model.named_parameters() if 'lora_' in n)
    print(f"UNI+PEFT-LoRA: {n_total:,} total params, {n_lora:,} LoRA params ({100*n_lora/n_total:.3f}%)")
    return model


def load_lora_state_into_model(model, ckpt_path):
    """Load the LoRA state from a Stage-2 training checkpoint.

    v3 fix: use plain `load_state_dict(strict=False)` — EXACTLY what training
    used in `_load_trainable_state(encoder, lora_state)`. The keys in
    `ckpt['lora_state']` are already `named_parameters()` keys, not PEFT's
    state-dict-with-suffixes format, so PEFT's `set_peft_model_state_dict`
    silently doesn't match them. Direct loading works because the keys are
    already in the form the model expects.
    """
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if 'lora_state' not in ckpt:
        raise RuntimeError(f"Checkpoint {ckpt_path} has no 'lora_state' key. Keys: {list(ckpt.keys())}")
    lora_state = ckpt['lora_state']

    # Sample BEFORE load: grab one lora_A param so we can detect that it changed
    sample_key = next(k for k in lora_state if 'lora_A' in k)
    sample_param = dict(model.named_parameters()).get(sample_key)
    if sample_param is None:
        # Key doesn't exist as a model param — show some model keys for debug
        model_lora_sample = [n for n, _ in model.named_parameters() if 'lora_' in n][:3]
        raise RuntimeError(
            f"Checkpoint key '{sample_key}' not found in model.\n"
            f"Model LoRA keys look like: {model_lora_sample}")
    before = sample_param.detach().cpu().clone()

    # Load via PLAIN state_dict — matches training-time _load_trainable_state
    incompatible = model.load_state_dict(lora_state, strict=False)
    # `incompatible.missing_keys` will be huge (all the frozen base) — that's expected
    # `incompatible.unexpected_keys` should be empty
    if incompatible.unexpected_keys:
        print(f"  WARNING: unexpected keys: {incompatible.unexpected_keys[:3]}...")

    # Verify: the sample param should now equal the checkpoint value
    after = sample_param.detach().cpu()
    ckpt_t = lora_state[sample_key]
    changed = not torch.allclose(before, after, atol=1e-10)
    matches_ckpt = torch.allclose(after, ckpt_t, atol=1e-6)

    n_ckpt = len(lora_state)
    n_model = len([n for n, _ in model.named_parameters() if 'lora_' in n])
    qwk = ckpt.get('qwk', None)
    fold = ckpt.get('fold', None)
    print(f"  Checkpoint LoRA tensors: {n_ckpt}")
    print(f"  Model LoRA params:       {n_model}")
    print(f"  Sample key '{sample_key.split('.')[-2:]}': changed={changed}, matches_ckpt={matches_ckpt}")
    print(f"  Checkpoint metadata:     fold={fold}, QWK={qwk:.4f}" if qwk else f"  Metadata: fold={fold}")
    if not matches_ckpt:
        raise RuntimeError(
            f"LoRA load FAILED: model param doesn't match checkpoint after load. "
            f"DO NOT proceed — features would be garbage.")
    return n_ckpt


# ══════════════════════════════════════════════════════════════════
# Multi-worker extraction — one fold at a time
# ══════════════════════════════════════════════════════════════════

def _extract_for_fold(fold, slide_ids, encoder, preprocessor, cfg, completed_set, failed_set, _lock):
    N_WORKERS = cfg['n_workers']
    LOCAL_TEMP = cfg['local_temp_dir']
    os.makedirs(LOCAL_TEMP, exist_ok=True)
    batch_size = cfg['encoder_batch_size']

    _gpu_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    _gpu_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    slide_ids_q = queue.Queue()
    for sid in slide_ids:
        slide_ids_q.put(sid)
    gpu_q = queue.Queue(maxsize=N_WORKERS)
    preprocess_errors = []

    def _cpu_worker():
        while True:
            try:
                slide_id = slide_ids_q.get_nowait()
            except Exception:
                return
            src_path = os.path.join(cfg['slides_dir'], f"{slide_id}{cfg['slide_ext']}")
            local_path = os.path.join(LOCAL_TEMP, f"{slide_id}{cfg['slide_ext']}")
            used_local = False
            try:
                # Drive open with retry-via-copy fallback
                slide = None
                for attempt in range(3):
                    try:
                        if attempt == 0:
                            slide = openslide.OpenSlide(src_path)
                        else:
                            # Copy to local SSD first (more reliable than direct Drive read)
                            if not os.path.exists(local_path):
                                shutil.copy2(src_path, local_path)
                                used_local = True
                            slide = openslide.OpenSlide(local_path)
                        break
                    except Exception as _e:
                        if attempt == 2:
                            raise
                        time.sleep(2)
                patches, coords = preprocessor.extract_patches_and_coords(slide)
                slide.close()
                if len(patches) < cfg['min_patches_per_slide']:
                    with _lock:
                        preprocess_errors.append((slide_id, f"only {len(patches)} patches"))
                    gpu_q.put((slide_id, None, None, local_path if used_local else None))
                    continue
                patch_tensors = torch.from_numpy(np.stack(patches))
                coords_tensor = torch.tensor(coords, dtype=torch.float32)
                del patches
                gpu_q.put((slide_id, patch_tensors, coords_tensor, local_path if used_local else None))
            except Exception as e:
                with _lock:
                    preprocess_errors.append((slide_id, str(e)))
                gpu_q.put((slide_id, None, None, local_path if used_local and os.path.exists(local_path) else None))

    actual_workers = min(N_WORKERS, len(slide_ids))
    cpu_threads = []
    for _ in range(actual_workers):
        t = threading.Thread(target=_cpu_worker, daemon=True)
        t.start()
        cpu_threads.append(t)

    fold_completed, fold_failed = [], []
    pbar = tqdm(total=len(slide_ids), desc=f"Fold {fold}", unit="slide")
    processed = 0
    start_time = time.time()

    while processed < len(slide_ids):
        try:
            item = gpu_q.get(timeout=300)
        except Exception:
            alive = sum(1 for t in cpu_threads if t.is_alive())
            print(f"\n  Queue timeout. {alive}/{actual_workers} workers alive")
            if alive == 0:
                break
            continue
        slide_id, patch_tensors, coords_tensor, local_path = item
        processed += 1
        if patch_tensors is None:
            fold_failed.append(slide_id)
            pbar.update(1)
            try:
                if local_path and os.path.exists(local_path):
                    os.remove(local_path)
            except Exception:
                pass
            continue
        try:
            all_features = []
            with torch.inference_mode(), torch.amp.autocast('cuda'):
                for b in range(0, len(patch_tensors), batch_size):
                    batch_raw = patch_tensors[b:b+batch_size]
                    batch_gpu = batch_raw.to(device).permute(0, 3, 1, 2).float() / 255.0
                    # v5: no resize -- EXTRACTION_CONFIG['patch_size']=224 already matches UNI input.
                    batch_gpu = (batch_gpu - _gpu_mean) / _gpu_std
                    feats = encoder(batch_gpu).float().cpu()
                    all_features.append(feats)
                    del batch_raw, batch_gpu
            features_cat = torch.cat(all_features, dim=0)
            output_path = os.path.join(cfg['output_dir'], f"{slide_id}.pt")
            torch.save({'features': features_cat, 'coords': coords_tensor, 'fold': fold}, output_path)
            fold_completed.append(slide_id)
            del patch_tensors, all_features, features_cat, coords_tensor
            elapsed = time.time() - start_time
            rate = (processed - len(fold_failed)) / max(elapsed / 60, 0.01)
            remaining = (len(slide_ids) - processed) / max(rate, 0.01)
            pbar.set_postfix(rate=f"{rate:.1f}/min", eta=f"{remaining:.0f}min")
        except Exception as e:
            print(f"\n  GPU ERROR {slide_id}: {e}")
            fold_failed.append(slide_id)
        finally:
            try:
                if local_path and os.path.exists(local_path):
                    os.remove(local_path)
            except Exception:
                pass
        pbar.update(1)

        if len(fold_completed) % cfg['checkpoint_every'] == 0 and len(fold_completed) > 0:
            with _lock:
                completed_set.update(fold_completed)
                failed_set.update(fold_failed)
                _save_progress(cfg, completed_set, failed_set)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    pbar.close()
    for t in cpu_threads:
        t.join(timeout=10)
    return fold_completed, fold_failed, preprocess_errors


def _save_progress(cfg, completed_set, failed_set):
    progress_path = os.path.join(cfg['progress_dir'], 'lora_extraction_progress.json')
    with open(progress_path, 'w') as f:
        json.dump({
            'completed': sorted(completed_set),
            'failed': sorted(failed_set),
            'timestamp': datetime.now().isoformat(),
        }, f)


def extract_lora_features():
    cfg = EXTRACTION_CONFIG
    progress_path = os.path.join(cfg['progress_dir'], 'lora_extraction_progress.json')
    if os.path.exists(progress_path):
        with open(progress_path, 'r') as f:
            progress = json.load(f)
        completed_set = set(progress.get('completed', []))
        failed_set = set(progress.get('failed', []))
    else:
        completed_set, failed_set = set(), set()

    if os.path.isdir(cfg['output_dir']):
        existing = {f.replace('.pt', '') for f in os.listdir(cfg['output_dir']) if f.endswith('.pt')}
        completed_set |= existing

    fold_slides = {}
    for fold in range(5):
        sub = labels_df[labels_df['kfold'] == fold]
        ids = sub['image_id'].astype(str).tolist()
        retry_failed = cfg.get('retry_failed', True)
        if retry_failed:
            todo = [sid for sid in ids if sid not in completed_set]
        else:
            todo = [sid for sid in ids if sid not in completed_set and sid not in failed_set]
        fold_slides[fold] = todo

    print("=" * 60)
    print("LoRA FEATURE EXTRACTION v2 (PEFT, held-out fold per slide)")
    print("=" * 60)
    total_todo = sum(len(v) for v in fold_slides.values())
    print(f"Already done: {len(completed_set)}")
    print(f"To process:   {total_todo}")
    print(f"Per-fold workload: {[len(fold_slides[f]) for f in range(5)]}")

    if total_todo == 0:
        print("\nAll slides already processed!")
        return

    print()
    # FINAL: skip pre-flight slide verification. Drive FUSE is flaky on direct
    # opens; the real CPU workers below handle this with a local-copy fallback.
    # We don't want a transient Drive hiccup to abort the whole extraction.
    print("Skipping pre-flight slide verification (CPU workers handle Drive flakiness).")
    print()

    encoder = build_uni_with_peft_lora(LORA_CONFIG).to(device).eval()
    preprocessor = SlidePreprocessor(
        patch_size=cfg['patch_size'],
        target_mag=cfg['target_magnification'],
        min_tissue_ratio=cfg['min_tissue_ratio'],
        sat_thresh=cfg.get('sat_thresh', 8),
        median_blur=cfg.get('median_blur', 7),
        morph_close=cfg.get('morph_close', 4),
        white_sat_thresh=cfg.get('white_sat_thresh', 5),
        black_rgb_thresh=cfg.get('black_rgb_thresh', 40),
    )

    _lock = threading.Lock()
    overall_start = time.time()
    all_preprocess_errors = []

    for fold in range(5):
        todo = fold_slides[fold]
        if not todo:
            print(f"\n--- Fold {fold}: nothing to do (already encoded) ---")
            continue
        ckpt = os.path.join(LORA_CHECKPOINTS_DIR, LORA_CKPT_PATTERN.format(fold=fold))
        if not os.path.exists(ckpt):
            print(f"\n--- Fold {fold}: SKIPPING (no checkpoint at {ckpt}) ---")
            continue
        print(f"\n--- Fold {fold}: loading {LORA_CKPT_PATTERN.format(fold=fold)} ({len(todo)} slides) ---")
        # CRITICAL: load LoRA. raises if 0 tensors match.
        load_lora_state_into_model(encoder, ckpt)
        encoder.eval()
        f_done, f_fail, f_pre_err = _extract_for_fold(
            fold, todo, encoder, preprocessor, cfg, completed_set, failed_set, _lock)
        completed_set.update(f_done)
        failed_set.update(f_fail)
        all_preprocess_errors.extend(f_pre_err)
        _save_progress(cfg, completed_set, failed_set)
        print(f"  Fold {fold} done: {len(f_done)} ok, {len(f_fail)} failed")

    elapsed = time.time() - overall_start
    print(f"\n{'=' * 60}")
    print(f"EXTRACTION COMPLETE")
    print(f"  Wall-clock: {elapsed/3600:.2f} hours")
    print(f"  Total done: {len(completed_set)}")
    print(f"  Total failed: {len(failed_set)}")
    print(f"  Output: {cfg['output_dir']}")
    if all_preprocess_errors:
        print(f"  Preprocess errors: {len(all_preprocess_errors)}")
        for sid, err in all_preprocess_errors[:5]:
            print(f"    - {sid}: {err}")
    print(f"{'=' * 60}")

    shutil.rmtree(cfg['local_temp_dir'], ignore_errors=True)
    del encoder
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def verify_slide_reading(slides_dir, slide_ext, n_samples=5):
    print("=" * 60)
    print("SLIDE READING VERIFICATION")
    print("=" * 60)
    slide_files = sorted([f for f in os.listdir(slides_dir) if f.endswith(slide_ext)])
    if not slide_files:
        print(f"  ERROR: No {slide_ext} files in {slides_dir}")
        return False
    import random
    random.seed(42)
    samples = random.sample(slide_files, min(n_samples, len(slide_files)))
    test_preprocessor = SlidePreprocessor()
    issues = []
    for fname in samples:
        fpath = os.path.join(slides_dir, fname)
        slide = openslide.OpenSlide(fpath)
        level, read_size, step_l0, actual_mag, obj_power = test_preprocessor.get_extraction_params(slide)
        level_dims = slide.level_dimensions[level]
        needs_resize = (read_size != 256)
        if actual_mag >= 19.0:
            status = "OK"
        elif actual_mag >= 10.0:
            status = "WARNING"
            issues.append(f"{fname}: low mag {actual_mag:.1f}x")
        else:
            status = "FAIL"
            issues.append(f"{fname}: very low mag {actual_mag:.1f}x")
        resize_note = f" -> resize {read_size}->256" if needs_resize else ""
        print(f"  {fname}: obj={obj_power}x, level={level} ({level_dims[0]}x{level_dims[1]}), eff={actual_mag:.1f}x{resize_note} [{status}]")
        slide.close()
    return len(issues) == 0


print("Pipeline ready (v2 — PEFT).")
print("  build_uni_with_peft_lora(): creates the model with PEFT LoRA structure")
print("  load_lora_state_into_model(): loads ckpt['lora_state'] via PEFT, aborts if 0 match")
print("  extract_lora_features(): processes all 5 folds, held-out per slide")


## 4. Run Extraction

**v2 verification step:** before processing slides, the code checks that LoRA weights actually loaded by hashing 2 sample tensors and comparing to the checkpoint. If 0 match, it aborts immediately (so no more corrupted features).

Expected output per fold:
```
--- Fold 0: loading lora_fold0_best.pt (1801 slides) ---
  Checkpoint LoRA tensors: 48
  Model LoRA params:       48
  Sample tensor match:     2/2     ← MUST be 2/2, not 0/2
  Checkpoint metadata:     fold=0, QWK=0.9593
Fold 0: 100%|█████| 1801/1801 [HH:MM<00:00, rate=YY/min]
```

## Pre-Extraction Verification (run this first)

Checks Drive mount, all paths, checkpoint validity, LoRA config match, GPU, PEFT, labels, slides, and existing features. Aborts with a clear error list if anything is wrong — better to find out now than 5 hours into extraction.

In [ ]:
# ══════════════════════════════════════════════════
# PRE-EXTRACTION VERIFICATION
# Run this cell AFTER cell 6 (pipeline definition) and BEFORE
# extract_lora_features() to catch any environment issues early.
# ══════════════════════════════════════════════════

import os
import torch

print("=" * 60)
print("PRE-EXTRACTION VERIFICATION")
print("=" * 60)

errors = []
warnings = []

# 1. Drive mount
mounted = os.path.exists('/content/drive/MyDrive')
print(f"\n[1] Drive mounted: {'✓' if mounted else '✗'}")
if not mounted:
    errors.append("Drive not mounted")

# 2. Critical paths
critical_paths = {
    "SLIDES_DIR":           SLIDES_DIR,
    "OUTPUT_DIR (will be created)": OUTPUT_DIR,
    "PROGRESS_DIR (will be created)": PROGRESS_DIR,
    "LORA_CHECKPOINTS_DIR": LORA_CHECKPOINTS_DIR,
    "LABELS_PATH":          LABELS_PATH,
}
print(f"\n[2] Critical paths:")
for label, path in critical_paths.items():
    ok = os.path.exists(path)
    print(f"  {'✓' if ok else '✗'} {label}: {path}")
    if not ok and "will be created" not in label:
        errors.append(f"Missing: {path}")

# 3. LoRA checkpoints — how many exist
print(f"\n[3] LoRA checkpoints:")
ckpt_status = {}
for fold in range(5):
    p = os.path.join(LORA_CHECKPOINTS_DIR, LORA_CKPT_PATTERN.format(fold=fold))
    exists = os.path.exists(p)
    ckpt_status[fold] = exists
    size_mb = os.path.getsize(p) / 1e6 if exists else 0
    print(f"  Fold {fold}: {'✓' if exists else '✗'} ({size_mb:.1f} MB)" if exists else f"  Fold {fold}: ✗ (not yet trained)")

n_ready = sum(ckpt_status.values())
print(f"\n  {n_ready}/5 folds ready for extraction")
if n_ready == 0:
    errors.append("No LoRA checkpoints found")
elif n_ready < 5:
    warnings.append(f"Only {n_ready}/5 folds trained — extraction will skip missing folds")

# 4. Verify a sample checkpoint loads correctly
if n_ready > 0:
    print(f"\n[4] Verifying checkpoint structure (testing fold {min(f for f, ok in ckpt_status.items() if ok)}):")
    test_fold = min(f for f, ok in ckpt_status.items() if ok)
    test_ckpt = os.path.join(LORA_CHECKPOINTS_DIR, LORA_CKPT_PATTERN.format(fold=test_fold))
    try:
        ckpt = torch.load(test_ckpt, map_location='cpu', weights_only=False)
        required_keys = ['lora_state', 'fold', 'qwk', 'lora_config']
        for k in required_keys:
            present = k in ckpt
            print(f"  {'✓' if present else '✗'} '{k}' in checkpoint")
            if not present:
                errors.append(f"Checkpoint missing key: {k}")
        if 'lora_state' in ckpt:
            n_lora = len(ckpt['lora_state'])
            print(f"  ✓ lora_state has {n_lora} tensors")
            sample_key = next(iter(ckpt['lora_state'].keys()))
            print(f"  ✓ Sample key format: {sample_key}")
            if 'base_model.model' not in sample_key:
                warnings.append("Key format doesn't look like PEFT — check that training used PEFT")
        if 'lora_config' in ckpt:
            cfg = ckpt['lora_config']
            print(f"  ✓ Training LoRA config: r={cfg.get('r')}, alpha={cfg.get('lora_alpha')}, target={cfg.get('target_modules')}")
            # Check config matches what this notebook expects
            if cfg.get('r') != LORA_CONFIG['r']:
                errors.append(f"r mismatch: ckpt={cfg.get('r')}, notebook={LORA_CONFIG['r']}")
            if cfg.get('lora_alpha') != LORA_CONFIG['lora_alpha']:
                errors.append(f"alpha mismatch: ckpt={cfg.get('lora_alpha')}, notebook={LORA_CONFIG['lora_alpha']}")
            if cfg.get('target_modules') != LORA_CONFIG['target_modules']:
                errors.append(f"target_modules mismatch: ckpt={cfg.get('target_modules')}, notebook={LORA_CONFIG['target_modules']}")
        if 'qwk' in ckpt:
            print(f"  ✓ Training QWK: {ckpt['qwk']:.4f}")
        if 'fold' in ckpt:
            assert ckpt['fold'] == test_fold, f"Fold mismatch in checkpoint!"
            print(f"  ✓ Fold tag matches filename ({test_fold})")
    except Exception as e:
        errors.append(f"Could not load test checkpoint: {e}")
        print(f"  ✗ Failed to load: {e}")

# 5. GPU
print(f"\n[5] GPU:")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  ✓ {gpu} ({vram_gb:.1f} GB VRAM)")
    if vram_gb < 15:
        warnings.append(f"Low VRAM ({vram_gb:.1f} GB) — may need to reduce encoder_batch_size")
else:
    errors.append("No GPU available — extraction will be impossibly slow on CPU")

# 6. PEFT version (must support load_state_dict-style loading)
print(f"\n[6] PEFT version:")
try:
    import peft
    print(f"  ✓ peft {peft.__version__}")
except Exception as e:
    errors.append(f"peft not installed: {e}")

# 7. Labels CSV is loadable
print(f"\n[7] Labels:")
try:
    import pandas as pd
    df = pd.read_csv(LABELS_PATH)
    required_cols = ['image_id', 'kfold', 'isup_grade']
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        errors.append(f"Labels CSV missing columns: {missing_cols}")
        print(f"  ✗ Missing columns: {missing_cols}")
    else:
        print(f"  ✓ {len(df)} rows, columns OK")
        print(f"  ✓ Per-fold counts: {df.groupby('kfold').size().to_dict()}")
except Exception as e:
    errors.append(f"Could not read labels CSV: {e}")

# 8. Slides directory has tiff files
print(f"\n[8] Slides:")
try:
    tiff_files = [f for f in os.listdir(SLIDES_DIR) if f.endswith('.tiff')]
    print(f"  ✓ Found {len(tiff_files)} .tiff files in {SLIDES_DIR}")
    if len(tiff_files) < 100:
        warnings.append(f"Only {len(tiff_files)} .tiff files — expected ~9128")
except Exception as e:
    errors.append(f"Could not list slides: {e}")

# 9. Existing features (for resume)
print(f"\n[9] Existing features:")
if os.path.exists(OUTPUT_DIR):
    existing = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.pt')]
    print(f"  {len(existing)} .pt files already in {OUTPUT_DIR}")
    if existing:
        print(f"  Will resume — already-done slides will be skipped")
else:
    print(f"  Output dir doesn't exist yet (will be created)")

# Summary
print(f"\n{'=' * 60}")
print(f"SUMMARY")
print(f"{'=' * 60}")
if errors:
    print(f"\n❌ {len(errors)} ERROR(S) — DO NOT RUN EXTRACTION YET:")
    for e in errors:
        print(f"  - {e}")
else:
    print(f"\n✅ No errors. Ready to run extract_lora_features()")

if warnings:
    print(f"\n⚠️  {len(warnings)} warning(s):")
    for w in warnings:
        print(f"  - {w}")

print()


In [4]:
# Run LoRA feature extraction
# Resumes automatically from lora_extraction_progress.json + existing .pt files

extract_lora_features()


UNI v1 FEATURE EXTRACTION (Multi-Worker Pipeline)
Total slides:      9128
Already done:      9128
Previously failed: 0
To process:        0
Output dir:        /content/drive/MyDrive/UNI_features/
CPU workers:       5
Tissue seg:        HSV saturation (sat>8, blur=7)
Patch filtering:   white(sat<5) + black(rgb<40)
Magnification:     target 20x (40x slides auto-resized)

All slides already processed!


## 5. Verify Extracted Features

Quick sanity check on a random extracted feature: shape, fold metadata, no NaNs.

In [5]:
# Verify extracted LoRA features
import random
feat_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.pt')]
print(f"Total features: {len(feat_files)}")

# Check 5 random files
random.seed(0)
for fname in random.sample(feat_files, min(5, len(feat_files))):
    p = os.path.join(OUTPUT_DIR, fname)
    d = torch.load(p, map_location='cpu', weights_only=False)
    feats = d['features']
    has_nan = torch.isnan(feats).any().item()
    print(f"  {fname}: features={tuple(feats.shape)}, coords={tuple(d['coords'].shape)}, fold={d.get('fold','?')}, NaN={has_nan}")

# Per-fold counts (fold tag in metadata)
fold_counts = {f: 0 for f in range(5)}
for fname in feat_files[:1000]:  # sample, not all
    d = torch.load(os.path.join(OUTPUT_DIR, fname), map_location='cpu', weights_only=False)
    f = d.get('fold', -1)
    if f in fold_counts:
        fold_counts[f] += 1
print(f"\nFold tag distribution (first 1000): {fold_counts}")


Total .pt features: 9128

Sample features:
  34b8869c4b4ba4fdfea0a9e579502c5f.pt: features=torch.Size([403, 1024]), coords=torch.Size([403, 2]), mean=-0.0067, std=1.1947
  0bf60b57d6fbd60feb3923848bdc720a.pt: features=torch.Size([285, 1024]), coords=torch.Size([285, 2]), mean=-0.0091, std=1.1943
  81c391aa0e95da9f8cb3419bce40236c.pt: features=torch.Size([503, 1024]), coords=torch.Size([503, 2]), mean=-0.0150, std=1.2280
  73df680f7884e13355a1f1fc2bacc3a4.pt: features=torch.Size([466, 1024]), coords=torch.Size([466, 2]), mean=-0.0112, std=1.2054
  69ed9b36a92904dcee7b78c338e5f4f4.pt: features=torch.Size([569, 1024]), coords=torch.Size([569, 2]), mean=-0.0150, std=1.2092

Progress file:
  Completed: 9128
  Failed: 0
  Total: 9128
  Last saved: 2026-03-01T06:48:38.271349
